# Baseline — Classify NLP Papers with Noisy Labels

**Competition:** text classification where a fraction of the **training labels are
wrong** (noisy). Learn a model that is robust to the label noise.

- **Task:** multi-class text classification
- **Metric:** accuracy (on clean test labels)
- **Kaggle link:** _TODO: add link_

**Approach:** TF-IDF + Logistic Regression first (linear models are fairly
noise-tolerant), then one round of **confident-learning style cleaning**: drop
training rows the model itself is most sure are mislabeled, and retrain.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
print(train.shape, test.shape, train["label"].value_counts().to_dict())

(4000, 3) (7600, 2) {0: 1000, 2: 1000, 1: 1000, 3: 1000}


In [2]:
model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2),
                      LogisticRegression(max_iter=2000))

# Out-of-fold predicted probabilities on the noisy training set
proba = cross_val_predict(model, train["text"], train["label"], cv=5,
                          method="predict_proba")
classes = np.unique(train["label"])
oof_pred = classes[np.argmax(proba, axis=1)]
noisy_cv = accuracy_score(train["label"], oof_pred)
print(f"CV accuracy vs NOISY labels: {noisy_cv:.4f} (not the real quality!)")

CV accuracy vs NOISY labels: 0.7670 (not the real quality!)


In [3]:
# Confident learning, one round: drop rows where the model confidently disagrees
p_given = proba[np.arange(len(train)), np.searchsorted(classes, train['label'])]
keep = p_given > np.quantile(p_given, 0.10)      # drop the 10% most suspicious rows
print(f"dropping {np.sum(~keep)} suspicious rows")

model.fit(train.loc[keep, "text"], train.loc[keep, "label"])
sub = pd.DataFrame({"id": test["id"], "label": model.predict(test["text"])})
sub.to_csv("submission.csv", index=False)
sub.head()

dropping 400 suspicious rows


,id,label
0,test_00000,0
1,test_00001,3
2,test_00002,3
3,test_00003,1
4,test_00004,3


## Ideas to improve

- Use **cleanlab** to estimate the label-noise matrix properly instead of a fixed 10%.
- Iterate: clean → retrain → clean again.
- Co-teaching: train two models, each keeps the samples the *other* finds easy.
- Fine-tune a small transformer with label smoothing + early stopping (don't let it
  memorize the noise).
